# 🏍️ Materi 1 — Pemanfaatan NLP untuk Ekstraksi Data Dokumen & Analisis Teks Otomatis
### Training NLP

**Masalahnya:** setiap hari perusahaan menerima banyak dokumen (invoice, laporan, email, feedback pelanggan). Membaca dan menyalin datanya secara **manual** itu lambat, membosankan, dan rawan salah ketik.

**Solusinya:** komputer bisa "membaca" teks dan **menarik data penting secara otomatis** — mengubah *teks bebas* menjadi *tabel data yang rapi*.

| Bagian | Yang Dipraktikkan |
|--------|-------------------|
| 1.1 | Mengenal regex: bahasa pola untuk mencari teks |
| 1.2 | Ekstraksi data dari satu dokumen invoice |
| 1.3 | Ekstraksi rincian barang dengan grup regex |
| 1.4 | **Batch processing**: mengolah banyak dokumen sekaligus |
| 1.5 | Analisis frekuensi kata dari feedback pelanggan |
| 1.6 | Analisis sentimen sederhana + rekap visual |

**Cara menjalankan:** jalankan sel berurutan dari atas ke bawah (`Shift + Enter`, atau *Runtime → Run all* di Google Colab).

In [ ]:
# ===== Persiapan: import library =====
import re                        # regex: mencari pola dalam teks
from collections import Counter  # menghitung frekuensi kata
import pandas as pd              # menampilkan data dalam bentuk tabel
import matplotlib.pyplot as plt  # membuat grafik

print("Library siap digunakan ✅")

## 1.1 Mengenal Regex — Bahasa Pola untuk Mencari Teks

**Regex (Regular Expression)** = cara memberi tahu komputer *"carikan teks yang bentuknya seperti ini"*.

Simbol dasar yang paling sering dipakai:

| Simbol | Arti | Contoh Pola | Cocok Dengan |
|--------|------|-------------|--------------|
| `\d` | satu digit angka (0–9) | `\d\d` | `26` |
| `\d{4}` | tepat 4 digit | `\d{4}` | `2026` |
| `\w` | huruf, angka, atau `_` | `\w+` | `Honda`, `MPX2` |
| `+` | satu kali atau lebih | `\d+` | `123456` |
| `?` | boleh ada, boleh tidak | `Rp\s?` | `Rp ` atau `Rp` |
| `[.-]` | salah satu karakter di kurung | `[\d.]+` | `2.750.000` |
| `( )` | **grup**: bagian yang ingin diambil | `(\d+) pcs` | mengambil angkanya saja |

Mari kita lihat cara kerjanya:

In [ ]:
contoh_teks = "Pesanan 250 unit dikirim tanggal 15/06/2026 dan 30 unit tanggal 21/06/2026."

# Cari semua tanggal dengan pola: 2 digit / 2 digit / 4 digit
print("Semua tanggal :", re.findall(r"\d{2}/\d{2}/\d{4}", contoh_teks))

# Cari semua jumlah unit: angka yang diikuti kata 'unit'
print("Semua jumlah  :", re.findall(r"\d+\s+unit", contoh_teks))

# Dengan GRUP ( ): ambil angkanya saja, tanpa kata 'unit'
print("Angkanya saja :", re.findall(r"(\d+)\s+unit", contoh_teks))

💡 **Intuisinya:** regex seperti "stempel pencari" — sekali polanya dibuat, komputer bisa menemukan **semua** teks yang bentuknya sama, secepat apa pun dan sebanyak apa pun dokumennya.

## 1.2 Ekstraksi Data dari Satu Dokumen Invoice

Sekarang kita terapkan pada dokumen bisnis sungguhan:

In [ ]:
# Contoh dokumen (teks tidak terstruktur) — bayangkan ini hasil scan/email
dokumen = """
PT ASTRA HONDA MOTOR
INVOICE PEMBELIAN SPAREPART

No. Invoice : INV-2026-00123
Tanggal     : 15/06/2026
Pelanggan   : Bengkel Maju Jaya
Email       : maju.jaya@email.com
Telepon     : 0812-3456-7890

Rincian:
- Kampas rem depan   2 pcs    Rp 150.000
- Oli mesin MPX-2    4 botol  Rp 220.000
- Filter udara       1 pcs    Rp 55.000

Total Pembayaran : Rp 2.750.000
"""

print(dokumen)

In [ ]:
# Ekstraksi data penting menggunakan pola regex
no_invoice = re.findall(r"INV-\d{4}-\d{5}", dokumen)                  # INV-4angka-5angka
tanggal    = re.findall(r"\d{2}/\d{2}/\d{4}", dokumen)                # 15/06/2026
email      = re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", dokumen)         # xxx@xxx.xxx
telepon    = re.findall(r"08\d{2}[-\s]?\d{4}[-\s]?\d{4}", dokumen)   # nomor HP Indonesia
nominal    = re.findall(r"Rp\s?[\d.]+\d", dokumen)                    # Rp + angka & titik

print("No. Invoice  :", no_invoice)
print("Tanggal      :", tanggal)
print("Email        :", email)
print("Telepon      :", telepon)
print("Semua nominal:", nominal)

In [ ]:
# Ubah hasil ekstraksi menjadi TABEL yang rapi (data terstruktur)
hasil = pd.DataFrame({
    "Informasi":       ["No. Invoice", "Tanggal", "Email", "Telepon", "Total Pembayaran"],
    "Hasil Ekstraksi": [no_invoice[0], tanggal[0], email[0], telepon[0], nominal[-1]],
})
hasil

# 💡 Hasil ini bisa langsung disimpan ke Excel:
# hasil.to_excel("hasil_ekstraksi.xlsx", index=False)

## 1.3 Ekstraksi Rincian Barang dengan Grup Regex
---
### ✨ Bonus: Ekstraksi Otomatis Data Penting dengan spaCy NER (Versi Lebih Presisi)
Model NER multibahasa bawaan kadang kurang akurat untuk format dokumen Indonesia (invoice, telepon, nominal rupiah). Karena itu, gunakan pendekatan hybrid: spaCy + EntityRuler + regex terarah.

Jika belum install spaCy dan modelnya, jalankan 1x:

!pip install spacy
!python -m spacy download xx_ent_wiki_sm

Contoh pemakaian (lebih stabil untuk dokumen invoice):


In [ ]:
import re
import spacy
import pandas as pd

# 1) Muat model multilingual untuk baseline NER
nlp = spacy.load("xx_ent_wiki_sm")

# 2) Tambahkan EntityRuler untuk pola dokumen yang sangat spesifik
# Aman saat cell dijalankan berulang (hindari duplikasi pipe)
if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([
    {"label": "INVOICE_ID", "pattern": [{"TEXT": {"REGEX": "^INV-\\d{4}-\\d{5}$"}}]},
    {"label": "DATE", "pattern": [{"TEXT": {"REGEX": "^\\d{2}/\\d{2}/\\d{4}$"}}]},
    {"label": "EMAIL", "pattern": [{"TEXT": {"REGEX": "^[\\w.+-]+@[\\w-]+\\.[\\w.]+$"}}]},
    {"label": "PHONE", "pattern": [{"TEXT": {"REGEX": "^08\\d{2}-\\d{4}-\\d{4}$"}}]},
    {"label": "MONEY", "pattern": [{"LOWER": "rp"}, {"TEXT": {"REGEX": "^[\\d.]+$"}}]}
])

doc_spacy = nlp(dokumen)

# 3) Post-processing: ambil field berbasis konteks baris (lebih presisi untuk invoice)
m_pelanggan = re.search(r"Pelanggan\s*:\s*(.+)", dokumen)
pelanggan = m_pelanggan.group(1).strip() if m_pelanggan else None
m_header = re.search(r"^PT.+$", dokumen, flags=re.MULTILINE)
header_org = m_header.group(0).strip() if m_header else None

ekstrak_hybrid = []
if header_org:
    ekstrak_hybrid.append(("Perusahaan", header_org, "regex-konteks"))
if pelanggan:
    ekstrak_hybrid.append(("Pelanggan", pelanggan, "regex-konteks"))

# 4) Tambahkan entitas dari spaCy (hasil model + ruler)
for ent in doc_spacy.ents:
    if ent.label_ in ["INVOICE_ID", "DATE", "EMAIL", "PHONE", "MONEY"]:
        ekstrak_hybrid.append((ent.label_, ent.text, "spaCy-ruler"))

tabel_entitas = pd.DataFrame(ekstrak_hybrid, columns=["Field", "Nilai", "Metode"]).drop_duplicates()
tabel_entitas

Grup `( )` memungkinkan kita mengambil **beberapa bagian sekaligus** dari satu baris — nama barang, jumlah, dan harga:

In [ ]:
# Pola: "- <nama barang>  <angka> <satuan>  Rp <harga>"
#         (.+?)  = nama barang (teks apa pun, secukupnya)
#         (\d+)  = jumlah
#         ([\d.]+) = harga (angka & titik)
pola_barang = r"-\s+(.+?)\s{2,}(\d+)\s+\w+\s+Rp\s?([\d.]+)"

rincian = re.findall(pola_barang, dokumen)
tabel_rincian = pd.DataFrame(rincian, columns=["Nama Barang", "Qty", "Harga Satuan (Rp)"])
tabel_rincian

✅ Tiga baris teks bebas → tabel rincian yang siap dihitung. Inilah dasar sistem **input data otomatis**.

## 1.4 Batch Processing — Mengolah Banyak Dokumen Sekaligus

Kekuatan sebenarnya dari otomasi: **pola yang sama dipakai berulang** untuk ratusan/ribuan dokumen. Kita simulasikan dengan 3 invoice:

In [ ]:
kumpulan_invoice = [
    "No. Invoice : INV-2026-00123\nTanggal : 15/06/2026\nPelanggan : Bengkel Maju Jaya\nTotal Pembayaran : Rp 2.750.000",
    "No. Invoice : INV-2026-00124\nTanggal : 18/06/2026\nPelanggan : AHASS Sentosa Motor\nTotal Pembayaran : Rp 1.430.000",
    "No. Invoice : INV-2026-00125\nTanggal : 21/06/2026\nPelanggan : Bengkel Berkah Jaya\nTotal Pembayaran : Rp 875.000",
]

def ekstrak_invoice(teks):
    """Mengekstraksi data penting dari SATU teks invoice."""
    return {
        "No. Invoice": re.search(r"INV-\d{4}-\d{5}", teks).group(),
        "Tanggal":     re.search(r"\d{2}/\d{2}/\d{4}", teks).group(),
        "Pelanggan":   re.search(r"Pelanggan : (.+)", teks).group(1),
        "Total":       re.search(r"Rp\s?[\d.]+\d", teks).group(),
    }

# Terapkan fungsi yang sama ke SEMUA invoice — inilah otomasi!
rekap = pd.DataFrame([ekstrak_invoice(inv) for inv in kumpulan_invoice])
rekap

💡 **Bayangkan:** kode di atas sama saja jika `kumpulan_invoice` berisi **1.000 dokumen** — waktu prosesnya tetap hitungan detik. Secara manual, 1.000 invoice bisa memakan waktu berhari-hari.

## 1.5 Analisis Frekuensi Kata — Apa yang Paling Sering Dibicarakan Pelanggan?

Kata yang sering muncul dalam kumpulan feedback = petunjuk **topik utama** yang dibicarakan pelanggan.

In [ ]:
feedback = [
    "Pelayanan bengkel sangat memuaskan dan cepat",
    "Antrian servis terlalu lama, saya kecewa",
    "Mekanik ramah, hasil servis bagus dan memuaskan",
    "Sparepart mahal tapi kualitas bagus",
    "Proses booking servis cepat dan mudah",
    "Ruang tunggu nyaman, pelayanan ramah",
    "Menunggu sparepart terlalu lama, kecewa dengan estimasi waktu",
    "Servis cepat, harga wajar, hasil memuaskan",
]

# Gabungkan semua feedback, ubah ke huruf kecil, pecah menjadi kata
semua_kata = " ".join(feedback).lower().split()

# Buang kata umum yang tidak bermakna (disebut "stopword")
stopword_sederhana = {"dan", "yang", "di", "saya", "tapi", "terlalu", "sangat", "dengan"}
kata_bermakna = [k for k in semua_kata if k not in stopword_sederhana]

frekuensi = Counter(kata_bermakna)
print("10 kata paling sering muncul:")
frekuensi.most_common(10)

In [ ]:
# Visualisasikan agar mudah dibaca oleh siapa pun
kata, jumlah = zip(*frekuensi.most_common(8))
plt.figure(figsize=(8, 4))
plt.bar(kata, jumlah, color="#990011")
plt.title("Kata Paling Sering Muncul di Feedback Pelanggan")
plt.ylabel("Jumlah kemunculan")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 1.6 Analisis Sentimen Sederhana + Rekap Visual

Ide dasarnya:
- Siapkan **kamus kata positif** dan **kamus kata negatif**
- `skor = jumlah kata positif − jumlah kata negatif`
- Skor > 0 → **POSITIF**, skor < 0 → **NEGATIF**, skor = 0 → **NETRAL**

In [ ]:
kata_positif = {"memuaskan", "cepat", "bagus", "ramah", "mudah", "puas",
                "baik", "mantap", "nyaman", "wajar", "senang"}
kata_negatif = {"lama", "kecewa", "mahal", "lambat", "buruk", "rusak",
                "jelek", "susah", "menunggu", "antri"}

def analisis_sentimen(teks):
    kata = teks.lower().split()
    skor = sum(k in kata_positif for k in kata) - sum(k in kata_negatif for k in kata)
    if skor > 0:
        return "POSITIF"
    elif skor < 0:
        return "NEGATIF"
    return "NETRAL"

hasil_sentimen = pd.DataFrame({
    "Feedback Pelanggan": feedback,
    "Sentimen": [analisis_sentimen(f) for f in feedback],
})
hasil_sentimen

In [ ]:
# Rekap: berapa persen pelanggan puas?
rekap_sentimen = hasil_sentimen["Sentimen"].value_counts()
print(rekap_sentimen)

plt.figure(figsize=(5, 4))
warna = {"POSITIF": "#2C8A4B", "NEGATIF": "#990011", "NETRAL": "#9AA0A6"}
plt.bar(rekap_sentimen.index, rekap_sentimen.values,
        color=[warna[s] for s in rekap_sentimen.index])
plt.title("Rekap Sentimen Feedback Pelanggan")
plt.ylabel("Jumlah feedback")
plt.tight_layout()
plt.show()

💡 **Catatan penting:** ini versi paling sederhana agar konsepnya terasa. Di dunia nyata, analisis sentimen memakai model *machine learning* / LLM yang memahami konteks ("tidak bagus" = negatif) — tetapi **alur kerjanya sama**: teks masuk → kesimpulan otomatis keluar.

---
### ✨ Bonus: Analisis Sentimen Otomatis dengan BERT/Transformers
Dibanding aturan sederhana, model transformer (BERT) bisa membaca konteks dan analisis sentimen lebih akurat. Untuk Bahasa Indonesia, gunakan model khusus Indonesia (bukan model default berbahasa Inggris).

Jalankan kode berikut untuk analisis sentimen otomatis menggunakan pipeline Huggingface transformers (butuh internet untuk ambil model pertama kali):

Jika library belum terpasang, install: !pip install transformers torch


In [ ]:
from transformers import pipeline

# Model sentimen Bahasa Indonesia
model_id = "w11wo/indonesian-roberta-base-sentiment-classifier"
sentiment_model = pipeline(
    task="sentiment-analysis",
    model=model_id,
    tokenizer=model_id
)

feedback_tes = [
    "Pelayanan sangat cepat dan memuaskan",
    "Hasil servis lama dan mekanik kurang ramah",
    "Harga sparepart wajar, ruang tunggu nyaman",
    "Servisnya tidak bagus dan saya kecewa"
]

hasil_senti = sentiment_model(feedback_tes)
for fb, res in zip(feedback_tes, hasil_senti):
    print(f"Feedback: {fb}\n  Prediksi: {res['label']} (confidence: {res['score']:.2f})\n")


---
# 🎯 Rangkuman & Latihan Mandiri — Materi 1

| Teknik | Fungsi Kunci | Kegunaan di Kantor |
|--------|--------------|--------------------|
| Ekstraksi pola | `re.findall()`, `re.search()` | Input data invoice/laporan otomatis |
| Grup regex | `( )` di dalam pola | Mengambil beberapa kolom sekaligus |
| Batch processing | fungsi + loop | Ribuan dokumen dengan satu pola |
| Frekuensi kata | `Counter` | Menemukan topik utama feedback |
| Sentimen kamus | fungsi skor sederhana | Monitoring kepuasan pelanggan |

### ✍️ Latihan Mandiri
1. Tambahkan pola regex untuk mengekstraksi **nama pelanggan** dari variabel `dokumen` di bagian 1.2 (petunjuk: lihat cara `Pelanggan` diambil di bagian 1.4).
2. Tambahkan satu invoice baru ke `kumpulan_invoice`, jalankan ulang — pastikan rekap otomatis bertambah.
3. Tambahkan 5 kata baru ke kamus sentimen, lalu uji dengan 3 feedback buatan Anda sendiri.
4. **Tantangan:** buat pola regex untuk nomor plat kendaraan (contoh: `B 1234 XYZ`).

➡️ **Lanjut ke Materi 2:** bagaimana AI benar-benar "memahami" teks — dari kata menjadi angka.

---
## 💡 Jawaban Latihan Mandiri

### Latihan 1 — Ekstraksi Nama Pelanggan

In [ ]:
# Latihan 1: Ekstraksi nama pelanggan dari variabel dokumen (bagian 1.2)
nama_pelanggan = re.search(r"Pelanggan\s*:\s*(.+)", dokumen).group(1).strip()
print("Nama Pelanggan :", nama_pelanggan)

### Latihan 2 — Tambah Invoice Baru

In [ ]:
# Latihan 2: Tambah satu invoice baru ke kumpulan_invoice
kumpulan_invoice.append(
    "No. Invoice : INV-2026-00126\nTanggal : 25/06/2026\nPelanggan : AHASS Prima Motor\nTotal Pembayaran : Rp 3.150.000"
)

# Jalankan ulang batch processing
rekap = pd.DataFrame([ekstrak_invoice(inv) for inv in kumpulan_invoice])
rekap

### Latihan 3 — Tambah Kata Sentimen & Uji Feedback Baru

In [ ]:
# Latihan 3: Tambah 5 kata baru ke kamus sentimen, lalu uji dengan 3 feedback buatan sendiri
kata_positif.update({"bersih", "responsif", "terjangkau", "ramah", "profesional"})
kata_negatif.update({"kotor", "lambat", "sulit", "berisik", "bau"})

feedback_baru = [
    "Servis di sini sangat profesional dan hasilnya bersih",
    "Bengkel kotor dan berisik, saya sulit parkir",
    "Harga terjangkau, mekanik responsif dan ramah",
]

hasil_baru = pd.DataFrame({
    "Feedback": feedback_baru,
    "Sentimen": [analisis_sentimen(f) for f in feedback_baru],
})
hasil_baru

### Latihan 4 — Pola Regex Plat Kendaraan

In [ ]:
# Latihan 4: Pola regex untuk nomor plat kendaraan Indonesia (contoh: B 1234 XYZ)
contoh_plat = "B 1234 XYZ, D 5678 ABC, AB 1234 CD, B 1 A"

# Pola: 1-2 huruf, spasi, 1-4 digit, spasi, 1-3 huruf
pola_plat = r"\b[A-Z]{1,2}\s\d{1,4}\s[A-Z]{1,3}\b"

hasil_plat = re.findall(pola_plat, contoh_plat)
print("Plat kendaraan ditemukan:", hasil_plat)